# Week 2 Day 4: Duffel Flight Booking Agent
Gradio Chat Interface:  ANTHROPIC see last markdown NOTE

Claude handles the conversation and decides when to call Duffel's sandbox API via tool use:
- Tools: search flights, inspect an offer, book it
- Booking uses sandbox Duffel Balance — nothing real is ever charged

**Requirements:**
- `ANTHROPIC_API_KEY` — your Anthropic API key
- `DUFFEL_TOKEN` — a Duffel sandbox token (starts with `duffel_test_`)

In [ ]:
import os
import json
import requests
import gradio as gr
from dotenv import load_dotenv
from anthropic import Anthropic

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
DUFFEL_TOKEN = os.getenv("DUFFEL_TOKEN")
    
if ANTHROPIC_API_KEY:
    print(f"Anthropic API Key exists and begins {ANTHROPIC_API_KEY[:7]}")
else:
    print("Anthropic API Key not set")

if DUFFEL_TOKEN:
    print(f"Duffel token exists and begins {DUFFEL_TOKEN[:12]}")
else:
    print("Duffel token not set")


In [ ]:

client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None
MODEL = "claude-sonnet-4-6"

In [ ]:
DUFFEL_BASE_URL = "https://api.duffel.com"

duffel_headers = {
    "Authorization": f"Bearer {DUFFEL_TOKEN}",
    "Duffel-Version": "v2",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

# Cache so Claude can reference offers by short index (1, 2, 3...) instead of full Duffel IDs
SESSION_STATE = {"last_offers": {}}

In [ ]:
SYSTEM_PROMPT = """You are a friendly flight-booking assistant backed by the Duffel sandbox API (test data only, no real bookings or charges).

Guidelines:
- Ask for any missing trip details (origin, destination, dates, passenger count) before searching.
- After search_flights, present the shortlist clearly with option numbers, airline, price, and departure time.
- Before booking, confirm the chosen option and total price with the user, and collect the passenger's full name, date of birth, email, and phone number.
- Always call get_offer_requirements before book_flight if you haven't already seen the offer's requirements in this conversation.
- Never invent flight data — only report what the tools return.
- Remind the user this is sandbox/test data, not a real booking, the first time you book something.
"""

## Duffel helper functions (the "tools")

These are plain Python functions that Claude will call via tool use.


In [ ]:
def duffel_post(path, payload):
    resp = requests.post(f"{DUFFEL_BASE_URL}{path}", headers=duffel_headers, data=json.dumps(payload))
    if not resp.ok:
        return {"error": resp.text}
    return resp.json()["data"]


def duffel_get(path):
    resp = requests.get(f"{DUFFEL_BASE_URL}{path}", headers=duffel_headers)
    if not resp.ok:
        return {"error": resp.text}
    return resp.json()["data"]


In [ ]:
def search_flights(origin, destination, departure_date, return_date=None, passengers=1):
    """Create an offer request and return a shortlist of the cheapest offers."""
    slices = [{"origin": origin, "destination": destination, "departure_date": departure_date}]
    if return_date:
        slices.append({"origin": destination, "destination": origin, "departure_date": return_date})

    payload = {
        "data": {
            "slices": slices,
            "passengers": [{"type": "adult"} for _ in range(int(passengers))],
            "cabin_class": "economy",
        }
    }

    result = duffel_post("/air/offer_requests", payload)
    if "error" in result:
        return result

    offers = sorted(result["offers"], key=lambda o: float(o["total_amount"]))[:8]

    SESSION_STATE["last_offers"] = {str(i + 1): o["id"] for i, o in enumerate(offers)}

    shortlist = []
    for i, offer in enumerate(offers):
        seg = offer["slices"][0]["segments"][0]
        shortlist.append({
            "option": i + 1,
            "airline": offer["owner"]["name"],
            "departs": seg["departing_at"],
            "price": f"{offer['total_amount']} {offer['total_currency']}",
        })

    return {"offer_request_id": result["id"], "options": shortlist}


In [ ]:
def get_offer_requirements(option_number):
    """Look up passenger field requirements for a previously returned option."""
    offer_id = SESSION_STATE["last_offers"].get(str(option_number))
    if not offer_id:
        return {"error": f"No cached offer for option {option_number}. Run search_flights first."}

    offer = duffel_get(f"/air/offers/{offer_id}")
    if "error" in offer:
        return offer

    return {
        "offer_id": offer_id,
        "total_amount": offer["total_amount"],
        "total_currency": offer["total_currency"],
        "passenger_ids": [p["id"] for p in offer["passengers"]],
    }


In [ ]:
def book_flight(option_number, given_name, family_name, born_on, email, phone_number, gender="m", title="mr"):
    """Create an order (book the flight) using Duffel Balance — sandbox only, no real charge."""
    offer_id = SESSION_STATE["last_offers"].get(str(option_number))
    if not offer_id:
        return {"error": f"No cached offer for option {option_number}. Run search_flights first."}

    offer = duffel_get(f"/air/offers/{offer_id}")
    if "error" in offer:
        return offer

    passenger_id = offer["passengers"][0]["id"]

    payload = {
        "data": {
            "type": "instant",
            "selected_offers": [offer_id],
            "payments": [{
                "type": "balance",
                "currency": offer["total_currency"],
                "amount": offer["total_amount"],
            }],
            "passengers": [{
                "id": passenger_id,
                "title": title,
                "gender": gender,
                "given_name": given_name,
                "family_name": family_name,
                "born_on": born_on,
                "email": email,
                "phone_number": phone_number,
            }],
        }
    }

    order = duffel_post("/air/orders", payload)
    if "error" in order:
        return order

    return {
        "order_id": order["id"],
        "booking_reference": order["booking_reference"],
        "total_paid": f"{order['total_amount']} {order['total_currency']}",
    }


## Tool schema

In [ ]:
TOOLS = [
    {
        "name": "search_flights",
        "description": "Search for flights between two airports on given dates. Returns a numbered shortlist of offers.",
        "input_schema": {
            "type": "object",
            "properties": {
                "origin": {"type": "string", "description": "3-letter IATA airport code, e.g. LHR"},
                "destination": {"type": "string", "description": "3-letter IATA airport code, e.g. JFK"},
                "departure_date": {"type": "string", "description": "YYYY-MM-DD"},
                "return_date": {"type": "string", "description": "YYYY-MM-DD, omit for one-way"},
                "passengers": {"type": "integer", "description": "Number of adult passengers, default 1"},
            },
            "required": ["origin", "destination", "departure_date"],
        },
    },
    {
        "name": "get_offer_requirements",
        "description": "Get passenger info requirements and price for a specific numbered option from the last search.",
        "input_schema": {
            "type": "object",
            "properties": {"option_number": {"type": "integer", "description": "The option number from search_flights results"}},
            "required": ["option_number"],
        },
    },
    {
        "name": "book_flight",
        "description": "Book a specific numbered offer from the last search using sandbox Duffel Balance (no real payment).",
        "input_schema": {
            "type": "object",
            "properties": {
                "option_number": {"type": "integer"},
                "given_name": {"type": "string"},
                "family_name": {"type": "string"},
                "born_on": {"type": "string", "description": "YYYY-MM-DD"},
                "email": {"type": "string"},
                "phone_number": {"type": "string", "description": "E.164 format, e.g. +442080160508"},
                "gender": {"type": "string", "enum": ["m", "f"], "default": "m"},
                "title": {"type": "string", "enum": ["mr", "ms", "mrs", "miss"], "default": "mr"},
            },
            "required": ["option_number", "given_name", "family_name", "born_on", "email", "phone_number"],
        },
    },
]

TOOL_FUNCTIONS = {
    "search_flights": search_flights,
    "get_offer_requirements": get_offer_requirements,
    "book_flight": book_flight,
}


## Claude tool-use loop

In [ ]:
def run_agent_turn(history_messages):
    """Runs Claude with tool use until it produces a final text reply."""
    messages = list(history_messages)

    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1500,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages,
        )

        if response.stop_reason != "tool_use":
            final_text = "".join(block.text for block in response.content if block.type == "text")
            return final_text, messages + [{"role": "assistant", "content": response.content}]

        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            func = TOOL_FUNCTIONS.get(block.name)
            result = func(**block.input) if func else {"error": f"Unknown tool {block.name}"}
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(result),
            })

        messages.append({"role": "user", "content": tool_results})


## Gradio chat interface

Running this cell launches the app inline (or in a browser tab, depending on your Jupyter setup).

In [ ]:
def chat_fn(message, history):
    if not client:
        return "⚠️ ANTHROPIC_API_KEY is not set. Set it in the config cell above and re-run."
    if not DUFFEL_TOKEN:
        return "⚠️ DUFFEL_TOKEN is not set. Set it in the config cell above and re-run."

    api_messages = []
    for h in history:
        role = "user" if h["role"] == "user" else "assistant"
        api_messages.append({"role": role, "content": h["content"]})
    api_messages.append({"role": "user", "content": message})

    reply, _ = run_agent_turn(api_messages)
    return reply


demo = gr.ChatInterface(
    fn=chat_fn,
    type="messages",
    title="✈️ Duffel Sandbox Flight Assistant",
    description=(
        "Chat naturally to search and book test flights via Duffel's sandbox API. "
        "Try: 'Find me a one-way flight from LHR to JFK on 2026-08-15'"
    ),
    examples=[
        "Find flights from LHR to JFK, departing 2026-08-15, returning 2026-08-22",
        "What's the cheapest option?",
        "Book option 1 for me",
    ],
)

demo.launch()


## Notes

- Offers are cached in `SESSION_STATE` and referenced by short numbers (1, 2, 3…) instead of full Duffel IDs, so the chat stays natural.
- The tool loop follows the standard Claude pattern: call `messages.create`, check `stop_reason == "tool_use"`, execute the matching Python function, feed the result back as a `tool_result`, repeat until Claude returns plain text.
- To stop the running Gradio server from a notebook, interrupt the kernel or call `demo.close()` in a new cell.

ANTHROPIC
- TOOLS: uses input_schema 
- run_agent_turn: tool_use, tool_return

OPENAI
- TOOLS: uses parameters
- run_agent_turn: tool_calls